In [1]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *

# Show columns and select data format
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

[TJN TOOLS: Data processing] Module loaded.
[TJN TOOLS: Other functions] Module loaded.
[TJN TOOLS: Paths] Module loaded. Sharepoint FOUND at /Users/mariocuendagarcia/Library/CloudStorage/OneDrive-SharedLibraries-TaxJusticeNetworkLtd


In [2]:
sotj_2021 = pd.read_csv('../output/tables/2024/Scaling_Mario/SOTJ_sample_countries_2021.csv')

sotj_2021_robustness_check = pd.read_csv('../output/tables/2024/Scaling_Mario/SOTJ_sample_countries_2021_robustness_check.csv')


In [3]:
# Keep iso_partner, negative_misalignment, positive_misalignment, reported_profit, etr_average_corrected, cit, tax_revenue_loss, tax_revenue_gain
sotj_2021 = sotj_2021[['iso_partner', 'negative_misalignment', 'positive_misalignment', 'reported_profit', 'etr_average_corrected', 'cit', 'tax_revenue_loss', 'tax_revenue_gain']]

sotj_2021

,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain
0,ABW,68.57,0.00,-10.55,0.24,0.25,17.14,0.00
1,AFG,86.43,3.07,-38.12,0.07,0.20,17.29,0.22
2,AGO,168.28,329.58,"3,353.05",0.36,0.25,42.07,120.01
3,AIA,1.82,0.00,-3.13,0.00,0.00,0.00,0.00
4,ALB,44.82,4.30,27.29,0.08,0.15,6.72,0.36
...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,34.05,0.08,0.10,0.50,0.27
207,YEM,165.19,2.03,16.45,0.45,0.20,33.04,0.91
208,ZAF,"4,556.62","18,883.48","55,421.65",0.14,0.28,"1,275.85","2,573.69"
209,ZMB,156.27,0.00,343.88,0.26,0.35,54.70,0.00


In [4]:
# Keep iso_partner, negative_misalignment, positive_misalignment, reported_profit, etr_average_corrected, cit, tax_revenue_loss, tax_revenue_gain
sotj_2021_robustness_check = sotj_2021_robustness_check[['iso_partner', 'negative_misalignment', 'positive_misalignment', 'reported_profit', 'etr_average_corrected', 'cit', 'tax_revenue_loss', 'tax_revenue_gain']]

sotj_2021_robustness_check

,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain
0,ABW,68.57,0.00,-10.55,0.24,0.25,17.14,0.00
1,AFG,86.43,3.07,-38.12,0.07,0.20,17.29,0.22
2,AGO,168.28,329.58,"3,353.05",0.36,0.30,50.48,120.01
3,AIA,1.82,0.00,-3.13,0.00,NaN,NaN,0.00
4,ALB,44.82,4.30,27.29,0.08,0.15,6.72,0.36
...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,34.05,0.08,NaN,NaN,0.27
207,YEM,165.19,2.03,16.45,0.45,0.20,33.04,0.91
208,ZAF,"4,556.62","18,883.48","55,421.65",0.14,0.28,"1,275.85","2,573.69"
209,ZMB,156.27,0.00,343.88,0.26,0.35,54.70,0.00


In [5]:
# In sotj_2021_robustness_check, generate reported_profit corrected as reported_profit - positive_misalignment
sotj_2021['reported_profit_corrected'] = sotj_2021['reported_profit'] - sotj_2021['positive_misalignment']
sotj_2021

,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain,reported_profit_corrected
0,ABW,68.57,0.00,-10.55,0.24,0.25,17.14,0.00,-10.55
1,AFG,86.43,3.07,-38.12,0.07,0.20,17.29,0.22,-41.18
2,AGO,168.28,329.58,"3,353.05",0.36,0.25,42.07,120.01,"3,023.48"
3,AIA,1.82,0.00,-3.13,0.00,0.00,0.00,0.00,-3.13
4,ALB,44.82,4.30,27.29,0.08,0.15,6.72,0.36,22.99
...,...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,34.05,0.08,0.10,0.50,0.27,30.65
207,YEM,165.19,2.03,16.45,0.45,0.20,33.04,0.91,14.42
208,ZAF,"4,556.62","18,883.48","55,421.65",0.14,0.28,"1,275.85","2,573.69","36,538.17"
209,ZMB,156.27,0.00,343.88,0.26,0.35,54.70,0.00,343.88


In [6]:

cit_2016 = sotj_2021_robustness_check[['iso_partner', 'cit']]

# Rename cit to cit_2016
cit_2016 = cit_2016.rename(columns={'cit': 'cit_2016'})

cit_2021 = sotj_2021[['iso_partner', 'cit']]

# Rename cit to cit_2021
cit_2021 = cit_2021.rename(columns={'cit': 'cit_2021'})

# Merge cit_2016 and cit_2021
cit_2016_2021 = pd.merge(cit_2016, cit_2021, on='iso_partner', how='inner')

# Generate cit_change as cit_2021 - cit_2016
cit_2016_2021['cit_change'] = cit_2016_2021['cit_2021'] - cit_2016_2021['cit_2016']

# If cit_change is positive, generate a new variable called cit_change_direction as 'increase'
# If cit_change is negative, generate a new variable called cit_change_direction as 'decrease'
# If cit_change is zero, generate a new variable called cit_change_direction as 'no_change'
cit_2016_2021['cit_change_direction'] = np.where(cit_2016_2021['cit_change'] > 0, 'increase', np.where(cit_2016_2021['cit_change'] < 0, 'decrease', 'no_change'))

cit_2016_2021

,iso_partner,cit_2016,cit_2021,cit_change,cit_change_direction
0,ABW,0.25,0.25,0.00,no_change
1,AFG,0.20,0.20,0.00,no_change
2,AGO,0.30,0.25,-0.05,decrease
3,AIA,NaN,0.00,NaN,no_change
4,ALB,0.15,0.15,0.00,no_change
...,...,...,...,...,...
206,XKV,NaN,0.10,NaN,no_change
207,YEM,0.20,0.20,0.00,no_change
208,ZAF,0.28,0.28,0.00,no_change
209,ZMB,0.35,0.35,0.00,no_change


In [ ]:
# merge sotj_2021 and cit_2016_2021
sotj_2021_both_cits = pd.merge(sotj_2021, cit_2016_2021, on='iso_partner', how='inner')

# If cit_change_direction is decrease, generate a new variable called lost_tax_revenues equal to reported_profit_corrected (only if above 0) * cit_2016
sotj_2021_both_cits['taxes_if_2016_cit'] = np.where(sotj_2021_both_cits['cit_change_direction'] == 'decrease', 
	np.where(sotj_2021_both_cits['reported_profit_corrected'] > 0, 
			 sotj_2021_both_cits['reported_profit_corrected'] * sotj_2021_both_cits['cit_2016'], 
			 0), 
	0)

sotj_2021_both_cits['taxes_if_2021_cit'] = np.where(sotj_2021_both_cits['cit_change_direction'] == 'decrease', 
	np.where(sotj_2021_both_cits['reported_profit_corrected'] > 0, 
			 sotj_2021_both_cits['reported_profit_corrected'] * sotj_2021_both_cits['cit_2021'], 
			 0), 
	0)

# Generate a new variable called tax_loss_race_bottom = lost_tax_revenues_2016 - lost_tax_revenues_2021
sotj_2021_both_cits['tax_loss_race_bottom'] = sotj_2021_both_cits['taxes_if_2016_cit'] - sotj_2021_both_cits['taxes_if_2021_cit']

sotj_2021_both_cits


,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain,reported_profit_corrected,cit_2016,cit_2021,cit_change,cit_change_direction,taxes_if_2016_cit,taxes_if_2021_cit,tax_loss_race_bottom
0,ABW,68.57,0.00,-10.55,0.24,0.25,17.14,0.00,-10.55,0.25,0.25,0.00,no_change,0.00,0.00,0.00
1,AFG,86.43,3.07,-38.12,0.07,0.20,17.29,0.22,-41.18,0.20,0.20,0.00,no_change,0.00,0.00,0.00
2,AGO,168.28,329.58,"3,353.05",0.36,0.25,42.07,120.01,"3,023.48",0.30,0.25,-0.05,decrease,907.04,755.87,151.17
3,AIA,1.82,0.00,-3.13,0.00,0.00,0.00,0.00,-3.13,NaN,0.00,NaN,no_change,0.00,0.00,0.00
4,ALB,44.82,4.30,27.29,0.08,0.15,6.72,0.36,22.99,0.15,0.15,0.00,no_change,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,34.05,0.08,0.10,0.50,0.27,30.65,NaN,0.10,NaN,no_change,0.00,0.00,0.00
207,YEM,165.19,2.03,16.45,0.45,0.20,33.04,0.91,14.42,0.20,0.20,0.00,no_change,0.00,0.00,0.00
208,ZAF,"4,556.62","18,883.48","55,421.65",0.14,0.28,"1,275.85","2,573.69","36,538.17",0.28,0.28,0.00,no_change,0.00,0.00,0.00
209,ZMB,156.27,0.00,343.88,0.26,0.35,54.70,0.00,343.88,0.35,0.35,0.00,no_change,0.00,0.00,0.00


In [8]:
# Filter by cit_change_direction = decrease
sotj_2021_decrease = sotj_2021_both_cits[sotj_2021_both_cits['cit_change_direction'] == 'decrease']

# Filter by cit_change_direction = increase
sotj_2021_increase = sotj_2021_both_cits[sotj_2021_both_cits['cit_change_direction'] == 'increase']

sotj_2021_increase

,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain,reported_profit_corrected,cit_2016,cit_2021,cit_change,cit_change_direction,taxes_if_2016_cit,taxes_if_2021_cit,tax_loss_race_bottom
17,BGD,"1,033.45",0.00,"1,212.55",0.28,0.33,335.87,0.00,"1,212.55",0.25,0.33,0.08,increase,0.00,0.00,0.00
34,CHL,"23,128.45","1,280.08","43,054.29",0.20,0.27,"6,244.68",259.02,"41,774.21",0.24,0.27,0.03,increase,0.00,0.00,0.00
49,DEU,"125,868.40",0.00,"193,276.43",0.15,0.30,"37,686.52",0.00,"193,276.43",0.30,0.30,0.00,increase,0.00,0.00,0.00
55,ECU,652.10,0.00,"1,968.51",0.30,0.25,163.03,0.00,"1,968.51",0.22,0.25,0.03,increase,0.00,0.00,0.00
103,KOR,"6,791.73",0.00,"20,375.26",0.23,0.28,"1,867.73",0.00,"20,375.26",0.24,0.28,0.03,increase,0.00,0.00,0.00
106,LBN,247.20,0.00,-97.26,0.22,0.17,42.02,0.00,-97.26,0.15,0.17,0.02,increase,0.00,0.00,0.00
115,LVA,593.97,6.22,435.70,0.06,0.20,118.79,0.39,429.48,0.15,0.20,0.05,increase,0.00,0.00,0.00
146,OMN,584.99,0.00,"1,042.67",0.30,0.15,87.75,0.00,"1,042.67",0.12,0.15,0.03,increase,0.00,0.00,0.00
149,PER,"5,458.35","1,331.77","26,248.70",0.23,0.29,"1,610.21",304.05,"24,916.93",0.28,0.29,0.01,increase,0.00,0.00,0.00
155,PRT,"7,119.08",602.45,"10,535.43",0.12,0.32,"2,242.51",71.89,"9,932.98",0.29,0.32,0.02,increase,0.00,0.00,0.00


In [9]:
# Sum tax_loss_race_bottom from sotj_2021_robustness_check_decrease
sum_race_bottom = sotj_2021_decrease['tax_loss_race_bottom'].sum()

sum_tax_loss = sotj_2021_decrease['tax_revenue_loss'].sum()

sum_tax_loss

137659.5038707001

In [12]:
### For Mark
sum_negative_misalignemnt_decreasers = sotj_2021_decrease['negative_misalignment'].sum()
sum_positive_misalignemnt_decreasers = sotj_2021_decrease['positive_misalignment'].sum()
sum_tax_loss_decreasers = sotj_2021_decrease['tax_revenue_loss'].sum()
sum_tax_increase_decreasers = sotj_2021_decrease['tax_revenue_gain'].sum()


sum_negative_misalignemnt_increasers = sotj_2021_increase['negative_misalignment'].sum()
sum_positive_misalignemnt_increasers = sotj_2021_increase['positive_misalignment'].sum()
sum_tax_loss_increasers = sotj_2021_increase['tax_revenue_loss'].sum()
sum_tax_increase_increasers = sotj_2021_increase['tax_revenue_gain'].sum()


# Create a dictionary with the variable names and their values
data = {
    'Variable': [
        'sum_negative_misalignemnt_decreasers', 
        'sum_positive_misalignemnt_decreasers', 
        'sum_tax_loss_decreasers', 
        'sum_tax_increase_decreasers', 
        'sum_negative_misalignemnt_increasers', 
        'sum_positive_misalignemnt_increasers', 
        'sum_tax_loss_increasers', 
        'sum_tax_increase_increasers'
    ],
    'Value': [
        sum_negative_misalignemnt_decreasers, 
        sum_positive_misalignemnt_decreasers, 
        sum_tax_loss_decreasers, 
        sum_tax_increase_decreasers, 
        sum_negative_misalignemnt_increasers, 
        sum_positive_misalignemnt_increasers, 
        sum_tax_loss_increasers, 
        sum_tax_increase_increasers
    ]
}

# Convert the dictionary to a DataFrame
estimates_df = pd.DataFrame(data)

# Display the DataFrame
estimates_df

# show me all the estimates above, generating a table with 8 rows and 2 columns, where column 1 is the name of the variable and column 2 is the value of the variable



,Variable,Value
0,sum_negative_misalignemnt_decreasers,"580,106.35"
1,sum_positive_misalignemnt_decreasers,"294,468.80"
2,sum_tax_loss_decreasers,"137,659.50"
3,sum_tax_increase_decreasers,"20,069.34"
4,sum_negative_misalignemnt_increasers,"177,814.73"
5,sum_positive_misalignemnt_increasers,"14,776.29"
6,sum_tax_loss_increasers,"51,846.45"
7,sum_tax_increase_increasers,"2,317.80"


*Playing with the 2016 exercise

0	rb_sum_negative_misalignemnt_decreasers	580,106.35
1	rb_sum_positive_misalignemnt_decreasers	294,468.80
2	rb_sum_tax_loss_decreasers	175,522.57
3	rb_sum_tax_increase_decreasers	20,069.34
4	rb_sum_negative_misalignemnt_increasers	177,814.73
5	rb_sum_positive_misalignemnt_increasers	14,776.29
6	rb_sum_tax_loss_increasers	50,143.08
7	rb_sum_tax_increase_increasers	2,317.80

In [13]:
# In sotj_2021_robustness_check, generate reported_profit corrected as reported_profit - positive_misalignment
sotj_2021_robustness_check['reported_profit_corrected'] = sotj_2021_robustness_check['reported_profit'] - sotj_2021_robustness_check['positive_misalignment']
sotj_2021_robustness_check

,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain,reported_profit_corrected
0,ABW,68.57,0.00,-10.55,0.24,0.25,17.14,0.00,-10.55
1,AFG,86.43,3.07,-38.12,0.07,0.20,17.29,0.22,-41.18
2,AGO,168.28,329.58,"3,353.05",0.36,0.30,50.48,120.01,"3,023.48"
3,AIA,1.82,0.00,-3.13,0.00,NaN,NaN,0.00,-3.13
4,ALB,44.82,4.30,27.29,0.08,0.15,6.72,0.36,22.99
...,...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,34.05,0.08,NaN,NaN,0.27,30.65
207,YEM,165.19,2.03,16.45,0.45,0.20,33.04,0.91,14.42
208,ZAF,"4,556.62","18,883.48","55,421.65",0.14,0.28,"1,275.85","2,573.69","36,538.17"
209,ZMB,156.27,0.00,343.88,0.26,0.35,54.70,0.00,343.88


In [14]:
# merge sotj_2021 and cit_2016_2021
sotj_2021_rb_both_cits = pd.merge(sotj_2021_robustness_check, cit_2016_2021, on='iso_partner', how='inner')

# If cit_change_direction is decrease, generate a new variable called lost_tax_revenues equal to reported_profit_corrected (only if above 0) * cit_2016
sotj_2021_rb_both_cits['taxes_if_2016_cit'] = np.where(sotj_2021_rb_both_cits['cit_change_direction'] == 'decrease', 
	np.where(sotj_2021_rb_both_cits['reported_profit_corrected'] > 0, 
			 sotj_2021_rb_both_cits['reported_profit_corrected'] * sotj_2021_rb_both_cits['cit_2016'], 
			 0), 
	0)

sotj_2021_rb_both_cits['taxes_if_2021_cit'] = np.where(sotj_2021_rb_both_cits['cit_change_direction'] == 'decrease', 
	np.where(sotj_2021_rb_both_cits['reported_profit_corrected'] > 0, 
			 sotj_2021_rb_both_cits['reported_profit_corrected'] * sotj_2021_rb_both_cits['cit_2021'], 
			 0), 
	0)

# Generate a new variable called tax_loss_race_bottom = lost_tax_revenues_2016 - lost_tax_revenues_2021
sotj_2021_rb_both_cits['tax_loss_race_bottom'] = sotj_2021_rb_both_cits['taxes_if_2016_cit'] - sotj_2021_rb_both_cits['taxes_if_2021_cit']

sotj_2021_rb_both_cits


,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain,reported_profit_corrected,cit_2016,cit_2021,cit_change,cit_change_direction,taxes_if_2016_cit,taxes_if_2021_cit,tax_loss_race_bottom
0,ABW,68.57,0.00,-10.55,0.24,0.25,17.14,0.00,-10.55,0.25,0.25,0.00,no_change,0.00,0.00,0.00
1,AFG,86.43,3.07,-38.12,0.07,0.20,17.29,0.22,-41.18,0.20,0.20,0.00,no_change,0.00,0.00,0.00
2,AGO,168.28,329.58,"3,353.05",0.36,0.30,50.48,120.01,"3,023.48",0.30,0.25,-0.05,decrease,907.04,755.87,151.17
3,AIA,1.82,0.00,-3.13,0.00,NaN,NaN,0.00,-3.13,NaN,0.00,NaN,no_change,0.00,0.00,0.00
4,ALB,44.82,4.30,27.29,0.08,0.15,6.72,0.36,22.99,0.15,0.15,0.00,no_change,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,34.05,0.08,NaN,NaN,0.27,30.65,NaN,0.10,NaN,no_change,0.00,0.00,0.00
207,YEM,165.19,2.03,16.45,0.45,0.20,33.04,0.91,14.42,0.20,0.20,0.00,no_change,0.00,0.00,0.00
208,ZAF,"4,556.62","18,883.48","55,421.65",0.14,0.28,"1,275.85","2,573.69","36,538.17",0.28,0.28,0.00,no_change,0.00,0.00,0.00
209,ZMB,156.27,0.00,343.88,0.26,0.35,54.70,0.00,343.88,0.35,0.35,0.00,no_change,0.00,0.00,0.00


In [16]:
# Filter by cit_change_direction = decrease
sotj_2021_rb_decrease = sotj_2021_rb_both_cits[sotj_2021_rb_both_cits['cit_change_direction'] == 'decrease']

# Filter by cit_change_direction = increase
sotj_2021_rb_increase = sotj_2021_rb_both_cits[sotj_2021_rb_both_cits['cit_change_direction'] == 'increase']

In [17]:
### For Mark
rb_sum_negative_misalignemnt_decreasers = sotj_2021_rb_decrease['negative_misalignment'].sum()
rb_sum_positive_misalignemnt_decreasers = sotj_2021_rb_decrease['positive_misalignment'].sum()
rb_sum_tax_loss_decreasers = sotj_2021_rb_decrease['tax_revenue_loss'].sum()
rb_sum_tax_increase_decreasers = sotj_2021_rb_decrease['tax_revenue_gain'].sum()


rb_sum_negative_misalignemnt_increasers = sotj_2021_rb_increase['negative_misalignment'].sum()
rb_sum_positive_misalignemnt_increasers = sotj_2021_rb_increase['positive_misalignment'].sum()
rb_sum_tax_loss_increasers = sotj_2021_rb_increase['tax_revenue_loss'].sum()
rb_sum_tax_increase_increasers = sotj_2021_rb_increase['tax_revenue_gain'].sum()


# Create a dictionary with the variable names and their values
data = {
    'Variable': [
        'rb_sum_negative_misalignemnt_decreasers', 
        'rb_sum_positive_misalignemnt_decreasers', 
        'rb_sum_tax_loss_decreasers', 
        'rb_sum_tax_increase_decreasers', 
        'rb_sum_negative_misalignemnt_increasers', 
        'rb_sum_positive_misalignemnt_increasers', 
        'rb_sum_tax_loss_increasers', 
        'rb_sum_tax_increase_increasers'
    ],
    'Value': [
        rb_sum_negative_misalignemnt_decreasers, 
        rb_sum_positive_misalignemnt_decreasers, 
        rb_sum_tax_loss_decreasers, 
        rb_sum_tax_increase_decreasers, 
        rb_sum_negative_misalignemnt_increasers, 
        rb_sum_positive_misalignemnt_increasers, 
        rb_sum_tax_loss_increasers, 
        rb_sum_tax_increase_increasers
    ]
}

# Convert the dictionary to a DataFrame
estimates_df = pd.DataFrame(data)

# Display the DataFrame
estimates_df

# show me all the estimates above, generating a table with 8 rows and 2 columns, where column 1 is the name of the variable and column 2 is the value of the variable



,Variable,Value
0,rb_sum_negative_misalignemnt_decreasers,"580,106.35"
1,rb_sum_positive_misalignemnt_decreasers,"294,468.80"
2,rb_sum_tax_loss_decreasers,"175,522.57"
3,rb_sum_tax_increase_decreasers,"20,069.34"
4,rb_sum_negative_misalignemnt_increasers,"177,814.73"
5,rb_sum_positive_misalignemnt_increasers,"14,776.29"
6,rb_sum_tax_loss_increasers,"50,143.08"
7,rb_sum_tax_increase_increasers,"2,317.80"
